# AI Model Experiment & Evaluation
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membandingkan dua pendekatan AI dalam menyelesaikan task sentiment analysis ulasan pelanggan:
1. Model klasik Machine Learning (Scikit-learn)
2. LLM API (Gemini)

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

> 🔧 **Catatan Penyesuaian Dataset**
>
> Notebook ini menggunakan dataset `customer_reviews_sentiment.csv` dengan kolom `review_text` dan `sentiment` (nilai: `positif`/`negatif`).
>
> Apabila dataset final berbeda dari yang digunakan saat ini, sesuaikan bagian berikut:
> - Nama file pada `pd.read_csv(...)` di Section 2
> - Nama kolom teks dan label pada Section 3 (saat ini: `review_text`, `sentiment`)
> - Label yang diminta pada prompt LLM di Section 5.2 (saat ini: `positif`/`negatif`)
> - Studi Kasus pada Assignment Brief, apabila domain data berbeda dari e-commerce

## 1. Problem Statement

_Tuliskan pemahamanmu terhadap use case di sini (objective, target/label, batasan/asumsi)._

## 2. Import Library & Load Dataset

In [3]:
# 🔧 Sesuaikan nama file jika dataset final berbeda
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
import os

df = pd.read_csv('../data/customer_reviews_sentiment.csv')
df.head()

print(df.shape)
print(df.isnull().sum())
print(df.duplicated().sum())
print(df['sentiment'].value_counts())

(200, 4)
review_id       0
product_name    0
review_text     0
sentiment       0
dtype: int64
0
sentiment
positif    110
negatif     90
Name: count, dtype: int64


In [12]:
print("Teks unik:", df['review_text'].nunique())

# apakah teks yang sama selalu punya label yang sama?
cek_label = df.groupby('review_text')['sentiment'].nunique()
print("Teks dengan label bertentangan:", (cek_label > 1).sum())

df_unik = df.drop_duplicates(subset='review_text').reset_index(drop=True)
print(df_unik.shape)
print(df_unik['sentiment'].value_counts())

Teks unik: 40
Teks dengan label bertentangan: 0
(40, 4)
sentiment
negatif    20
positif    20
Name: count, dtype: int64


## 3. Menyiapkan Train/Test Split

Pastikan test set yang sama digunakan untuk kedua pendekatan agar perbandingan adil.

In [13]:
# 🔧 Sesuaikan nama kolom jika dataset final menggunakan nama kolom berbeda
X = df_unik['review_text']
y = df_unik['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))
print(y_test.value_counts())

Train size: 28
Test size: 12
sentiment
negatif    6
positif    6
Name: count, dtype: int64


## 4. Pendekatan 1 — Model Klasik (Scikit-learn)

### 4.1 Preprocessing & Feature Extraction

In [14]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

### 4.2 Training Model

In [15]:
clf = LogisticRegression()
clf.fit(X_train_vec, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

### 4.3 Prediksi pada Test Set

In [16]:
y_pred_classic = clf.predict(X_test_vec)

## 5. Pendekatan 2 — LLM API (Gemini)

### 5.1 Setup API

In [4]:
# Simpan API key di environment variable, JANGAN hardcode langsung di notebook
from dotenv import load_dotenv
load_dotenv('../.env')

client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))
print("key aman:", os.environ.get('GEMINI_API_KEY') is not None)

key aman: True


In [22]:
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [5]:
MODEL_NAME = 'gemini-3.5-flash-lite'

r = client.models.generate_content(
    model=MODEL_NAME,
    contents='kamu siapa? jawab singkat dengan maksimal 10 kata'
)
print(r.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Saya adalah asisten AI yang siap membantu Anda.


### 5.2 Merancang Prompt

_Tuliskan prompt yang kamu rancang di sini, dan jelaskan alasannya (zero-shot / few-shot)._

In [6]:
def classify_sentiment_llm(review_text):
    prompt = f'''Kamu adalah sistem klasifikasi sentimen ulasan produk e-commerce berbahasa Indonesia.
Klasifikasikan sentimen ulasan berikut sebagai "positif" atau "negatif".
Jawab hanya dengan satu kata: positif atau negatif. Jangan beri penjelasan.

Ulasan: "{review_text}"
Sentimen:'''
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.0)
    )
    return response.text.strip().lower()

In [17]:
for teks in X_test.iloc[:3]:
    print(teks, '->', classify_sentiment_llm(teks))

Komplain tidak ditanggapi sama sekali oleh penjual. -> negatif
Respon penjual lambat dan terkesan tidak peduli komplain. -> negatif
Puas banget belanja disini, proses cepat dan barang aman. -> positif


### 5.3 Menjalankan Prediksi pada Test Set

Catatan: lakukan normalisasi terhadap output LLM sebelum dibandingkan dengan label asli (misal: lowercase, strip whitespace).

In [18]:
y_pred_llm = ['invalid'] * len(X_test)
raw_outputs = ['belum dijalankan'] * len(X_test)
latencies = []
print("ready:", len(y_pred_llm), "baris")

ready: 12 baris


In [ ]:
import time

def normalisasi(teks):
    t = teks.strip().lower().strip('."\'!, ')
    if t in ('positif', 'negatif'):
        return t
    if 'negatif' in t and 'positif' not in t:
        return 'negatif'
    if 'positif' in t and 'negatif' not in t:
        return 'positif'
    return 'invalid'

gagal_idx = [i for i, p in enumerate(y_pred_llm) if p == 'invalid']
print("Ulasan yang perlu diproses:", gagal_idx)

stop = False
for i in gagal_idx:
    if stop:
        break
    review = X_test.iloc[i]
    raw = 'error'
    for percobaan in range(3):
        try:
            mulai = time.time()
            raw = classify_sentiment_llm(review)   # <-- pemanggilan fungsinya
            latencies.append(time.time() - mulai)
            break
        except Exception as e:
            pesan = str(e)
            if 'PerDay' in pesan:
                print("Kuota HARIAN habis.")
                stop = True
                break
            elif '429' in pesan:
                print(f"Ulasan {i}: limit per menit. wait")
                time.sleep(45)
            elif '503' in pesan or '10054' in pesan:
                print(f"Ulasan {i}: server sibuk/koneksi putus")
                time.sleep(15)
            else:
                print(f"Ulasan {i} gagal: {e}")
                break
    if stop:
        break
    raw_outputs[i] = raw
    y_pred_llm[i] = normalisasi(raw)
    print(f"[ulasan {i}] {raw}")
    time.sleep(13)

print("Output invalid tersisa:", y_pred_llm.count('invalid'))
if latencies:
    print(f"Rata-rata waktu per panggilan: {sum(latencies)/len(latencies):.2f} detik")

pd.DataFrame({
    'review_text': X_test.values,
    'label_asli': y_test.values,
    'gemini_raw': raw_outputs,
    'gemini_pred': y_pred_llm
}).to_csv('../documentation/gemini_predictions.csv', index=False)

Ulasan yang perlu diproses: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
[ulasan 0] negatif
[ulasan 1] negatif
[ulasan 2] positif
[ulasan 3] negatif
[ulasan 4] positif
[ulasan 5] positif
[ulasan 6] positif
[ulasan 7] negatif
[ulasan 8] negatif
[ulasan 9] positif
[ulasan 10] positif
[ulasan 11] negatif
Output invalid tersisa: 0
Rata-rata waktu per panggilan: 15.07 detik


In [21]:
cek = pd.DataFrame({
    'review_text': X_test.values,
    'label_asli': y_test.values,
    'gemini_raw': raw_outputs,
    'gemini_pred': y_pred_llm
})
cek

,review_text,label_asli,gemini_raw,gemini_pred
0,Komplain tidak ditanggapi sama sekali oleh pen...,negatif,negatif,negatif
1,Respon penjual lambat dan terkesan tidak pedul...,negatif,negatif,negatif
2,"Puas banget belanja disini, proses cepat dan b...",positif,positif,positif
3,"Barang yang datang tidak sesuai dengan foto, k...",negatif,negatif,negatif
4,"Harga worth it dengan kualitas yang didapat, s...",positif,positif,positif
5,"Sudah langganan di toko ini, kualitas selalu k...",positif,positif,positif
6,"Kualitas premium dengan harga bersahabat, reco...",positif,positif,positif
7,"Pengiriman lambat sekali, sudah dua minggu bel...",negatif,negatif,negatif
8,"Ukuran dan warna berbeda dari pesanan, sangat ...",negatif,negatif,negatif
9,"Terima kasih seller, barangnya awet dan sesuai...",positif,positif,positif


## 6. Evaluasi dan Perbandingan

### 6.1 Evaluasi Model Klasik

In [15]:
print('=== Model Klasik (Scikit-learn) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_classic))
print(classification_report(y_test, y_pred_classic))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_classic))

=== Model Klasik (Scikit-learn) ===
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00         6
     positif       1.00      1.00      1.00         6

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12

Confusion Matrix:
 [[6 0]
 [0 6]]


In [16]:
hasil = pd.DataFrame({
    'review_text': X_test.values,
    'label_asli': y_test.values,
    'prediksi': y_pred_classic
})
salah = hasil[hasil['label_asli'] != hasil['prediksi']]
print("Jumlah salah prediksi:", len(salah))
salah

print(df['review_text'].duplicated().sum())

Jumlah salah prediksi: 0
160


In [19]:
pd.set_option('display.max_colwidth', None)
df_unik[['review_text', 'sentiment']]

,review_text,sentiment
0,Pelayanan lambat dan tidak responsif saat dikomplain.,negatif
1,"Puas banget belanja disini, proses cepat dan barang aman.",positif
2,"Terima kasih seller, barangnya awet dan sesuai deskripsi.",positif
3,Warna produk berbeda jauh dari foto di listing.,negatif
4,"Packing rapi, produk sesuai deskripsi. Recommended seller!",positif
5,Sudah bayar mahal tapi kualitas jauh dari ekspektasi.,negatif
6,"Belanja disini selalu puas, kualitas terjaga.",positif
7,"Bahan nyaman dipakai, ukuran pas, tidak mengecewakan.",positif
8,"Barang cacat produksi, jahitan lepas di beberapa bagian.",negatif
9,"Barang yang datang tidak sesuai dengan foto, kecewa berat.",negatif


In [20]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(TfidfVectorizer(), LogisticRegression())
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skor = cross_val_score(pipe, df_unik['review_text'], df_unik['sentiment'], cv=cv, scoring='f1_macro')
print("F1 per fold:", skor)
print("Rata-rata:", skor.mean())

F1 per fold: [0.75       0.87301587 0.87301587 0.36507937 0.56363636]
Rata-rata: 0.684949494949495


**Interpretasi Model Klasik:**
- Pada test set (12 ulasan), model mencapai Accuracy dan F1 = 1.00
  dengan confusion matrix tanpa kesalahan.
- Namun test set sangat kecil, sehingga hasil ini tidak stabil.
  Cross-validation 5-fold menunjukkan F1 macro rata-rata 0.68
  (rentang 0.37-0.87), sehingga performa sebenarnya kemungkinan
  jauh di bawah 1.00.
- Penyebab yang mungkin: hanya 28 data latih, dan TF-IDF tidak
  menangkap negasi (mis. "tidak mengecewakan" vs "mengecewakan").
- Sebelum deduplikasi, akurasi 1.00 juga muncul karena data leakage
  (160 dari 200 baris adalah duplikat).

### 6.2 Evaluasi LLM API

In [22]:
print('=== LLM API (Gemini) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_llm))
print(classification_report(y_test, y_pred_llm))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_llm))

=== LLM API (Gemini) ===
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00         6
     positif       1.00      1.00      1.00         6

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12

Confusion Matrix:
 [[6 0]
 [0 6]]


### 6.3 Tabel Perbandingan Ringkasan

_Susun tabel ringkasan (bisa markdown table atau DataFrame) yang membandingkan Accuracy, Precision, Recall, F1-Score kedua pendekatan._

In [23]:
import time

mulai = time.time()
_ = clf.predict(vectorizer.transform(X_test))
waktu_klasik = time.time() - mulai

print(f"Model klasik: {waktu_klasik*1000:.1f} ms untuk {len(X_test)} ulasan "
      f"({waktu_klasik/len(X_test)*1000:.2f} ms per ulasan)")
print(f"Gemini: {sum(latencies)/len(latencies):.2f} detik per ulasan")

Model klasik: 2.8 ms untuk 12 ulasan (0.24 ms per ulasan)
Gemini: 15.07 detik per ulasan


**Interpretasi Tabel Perbandingan:**
- Kedua pendekatan mencapai Accuracy, Precision, Recall, dan F1 = 1.00
  pada test set yang sama (12 ulasan, 6 positif dan 6 negatif).
  jadi sama sama bisa presisi
- Test set sangat kecil, sehingga skor sempurna belum bisa dianggap
  performa umum. Cross-validation model klasik menunjukkan F1 macro
  rata-rata 0.68 (rentang 0.37-0.87), artinya skor 1.00 di test set
  sebagian besar kebetulan. Gemini tidak dievaluasi dengan cara yang sama
  (hanya 12 ulasan), sehingga stabilitasnya belum terukur.
- Perbedaan jelas ada pada waktu: model klasik memprediksi dalam
  milidetik, sedangkan Gemini rata-rata 15.07 detik per ulasan.

## 7. Analisis Trade-off dan Limitation

_Tuliskan analisis: pendekatan mana yang lebih unggul dari sisi performa, trade-off effort/kecepatan/biaya, dan keterbatasan masing-masing pendekatan._

In [25]:
from sklearn.metrics import precision_recall_fscore_support

def ringkas(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {'Accuracy': accuracy_score(y_true, y_pred),
            'Precision (macro)': p, 'Recall (macro)': r, 'F1 (macro)': f}

tabel = pd.DataFrame({
    'Model Klasik (TF-IDF + LogReg)': ringkas(y_test, y_pred_classic),
    'Gemini (zero-shot)': ringkas(y_test, y_pred_llm),
}).T.round(3)

tabel['Waktu per ulasan'] = [
    f"{waktu_klasik/len(X_test)*1000:.2f} ms",
    f"{sum(latencies)/len(latencies):.2f} detik"
]
tabel

,Accuracy,Precision (macro),Recall (macro),F1 (macro),Waktu per ulasan
Model Klasik (TF-IDF + LogReg),1.0,1.0,1.0,1.0,0.24 ms
Gemini (zero-shot),1.0,1.0,1.0,1.0,15.07 detik


### Performa
Kedua pendekatan sama sama mendapat (1.00) pada test set 12 ulasan. Karena itu performa
tidak bisa dijadikan dasar utama pemilihan. Model klasik terbukti kurang
stabil pada data kecil (CV F1 rata-rata 0.68), sedangkan Gemini tidak
bergantung pada data latih sehingga kemungkinan lebih andal pada dataset
sekecil ini, tetapi bukti yang tersedia terbatas.

### Trade-off
| Aspek | Model Klasik | Gemini API |
|---|---|---|
| Implementasi | Perlu training | Tanpa training, cukup prompt |
| Kecepatan | X ms per ulasan | 15,07 detik per ulasan |
| Biaya | Gratis | Berbayar per panggilan |
| Koneksi | Lokal, tanpa internet | Perlu internet dan API key |
| Stabilitas | Konsisten | Bisa error atau model dihentikan |

### Limitation
- Dataset asli 200 baris, tetapi hanya 40 ulasan unik (160 duplikat).
  Sebelum deduplikasi, model klasik mendapat 1.00 karena data leakage.
- Test set hanya 12 ulasan, satu kesalahan setara turun sekitar 8%.
- Ulasan tergolong mudah diprediksi,belum menggunakan kata kompleks,
  campuran bahasa, atau ulasan netral, sehingga belum ada contoh salah
  prediksi untuk dianalisis.
- Model klasik berbasis TF-IDF tidak menangkap negasi dan konteks.
- Gemini: free tier dibatasi permintaan per menit dan batas harian,
  sehingga latency tinggi.

## 8. Rekomendasi Technical Approach

_Tuliskan rekomendasi akhir beserta alasannya, berdasarkan hasil eksperimen di atas._

**Rekomendasi: gunakan Gemini API untuk tahap awal, dan siapkan
transisi ke model klasik setelah data berlabel bertambah.**

Alasan:
1. Skor kedua pendekatan seri, sehingga keputusan bertumpu pada trade-off.
2. Data berlabel yang tersedia sangat kecil yaitu 40 ulasan unik. Pada kondisi
   ini model klasik tidak stabil (CV F1 0.68), sedangkan Gemini tidak butuh
   data latih dan langsung berfungsi.
3. Untuk volume kecil sampai menengah, waktu 15 detik per ulasan masih bisa
   ditangani lewat pemrosesan di belakang layar (batch atau antrean) dan
   hasilnya disimpan.

Syarat dan risikonya:
- Untuk trafik besar atau kebutuhan real-time, Gemini terlalu lambat dan
  biayanya naik seiring jumlah ulasan.
- Perlu paket berbayar, penanganan error (retry), dan pemantauan model
  yang dihentikan.

Langkah lanjutan: kumpulkan data berlabel lebih banyak (memakai hasil
Gemini yang dikoreksi manusia), lalu latih ulang model klasik dan
evaluasi ulang dengan test set yang lebih besar. Jika performanya setara,
pindah ke model klasik karena jauh lebih cepat dan murah.